# 05 · EDA con SQL
**Proyecto final – Capstone de Ciencia de Datos Aplicada (IBM)** · Autora: Jhoselyn Vera

**Objetivo:** consultar el historial de lanzamientos con SQL (SQLite en memoria, cargado con pandas).

In [ ]:
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
import sqlite3

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

URL = ("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
       "IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
raw = pd.read_csv(URL)
raw.columns = [c.strip() for c in raw.columns]
raw = raw.rename(columns={"Time (UTC)": "Time_UTC"})

# Fechas a formato ISO (AAAA-MM-DD) para poder filtrarlas con SQL
try:
    raw["Date"] = pd.to_datetime(raw["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d")
except ValueError:
    raw["Date"] = pd.to_datetime(raw["Date"], dayfirst=True).dt.strftime("%Y-%m-%d")

conn = sqlite3.connect(":memory:")
raw.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql_query(sql, conn)

q("SELECT * FROM SPACEXTBL LIMIT 5")

## Consulta 1 · Sitios de lanzamiento únicos

In [ ]:
q("SELECT DISTINCT Launch_Site FROM SPACEXTBL")

## Consulta 2 · Cinco registros de sitios que empiezan con 'CCA'

In [ ]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

## Consulta 3 · Masa total de carga útil de misiones de la NASA (CRS)

In [ ]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer LIKE 'NASA (CRS)%'")

## Consulta 4 · Masa promedio de carga útil del booster F9 v1.1

In [ ]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

## Consulta 5 · Fecha del primer aterrizaje exitoso en plataforma terrestre

In [ ]:
q("SELECT MIN(Date) AS first_ground_pad_success FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (ground pad)%'")

## Consulta 6 · Boosters con aterrizaje exitoso en barcaza y carga entre 4000 y 6000 kg

In [ ]:
q('''SELECT DISTINCT Booster_Version FROM SPACEXTBL
     WHERE Landing_Outcome LIKE 'Success (drone ship)%'
       AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000''')

## Consulta 7 · Total de misiones exitosas y fallidas

In [ ]:
q("SELECT Mission_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Mission_Outcome")

## Consulta 8 · Boosters que transportaron la carga máxima (subconsulta)

In [ ]:
q('''SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL
     WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)''')

## Consulta 9 · Aterrizajes fallidos en barcaza durante 2015

In [ ]:
q('''SELECT Date, Booster_Version, Launch_Site, Landing_Outcome FROM SPACEXTBL
     WHERE Landing_Outcome LIKE 'Failure (drone ship)%' AND substr(Date, 1, 4) = '2015' ''')

## Consulta 10 · Ranking de resultados de aterrizaje (2010-06-04 a 2017-03-20)

In [ ]:
q('''SELECT Landing_Outcome, COUNT(*) AS total FROM SPACEXTBL
     WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
     GROUP BY Landing_Outcome ORDER BY total DESC''')